# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("dataset/07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6378
Epoch 10/10 - Loss: 0.4077


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.750


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.64      0.71        96
    Positive       0.72      0.86      0.78       104

    accuracy                           0.75       200
   macro avg       0.76      0.75      0.75       200
weighted avg       0.76      0.75      0.75       200

Confusion Matrix:
[[61 35]
 [15 89]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [12]:
import numpy as np
import gensim
from nltk.tokenize import word_tokenize

# tokenize train and test reviews (same split as before)
train_tokens = [word_tokenize(review) for review in X_train]
test_tokens = [word_tokenize(review) for review in X_test]

print(train_tokens[:3])

[['the', 'worst', 'was', 'the', 'salmon', 'sashimi'], ['an', 'excellent', 'new', 'restaurant', 'by', 'an', 'experienced', 'frenchman'], ['went', 'for', 'lunch', 'service', 'was', 'slow']]


The dataset is small (only 800 training reviews), so with the default 5 epochs the word vectors were almost the same for every word. I set `epochs=50` so Word2Vec trains more on the same data. `seed` and `workers=1` are just so I get the same result every run.

In [13]:
w2v_model = gensim.models.Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1,
                                   epochs=50, seed=42, workers=1)

print("Vocabulary size:", len(w2v_model.wv))
w2v_model.wv.most_similar('good')

Vocabulary size: 1800


[('prices', 0.8068937063217163),
 ('affordable', 0.7989750504493713),
 ('reasonable', 0.7983586192131042),
 ('outstanding', 0.7907360792160034),
 ('exceptional', 0.7906337976455688),
 ('liked', 0.787510097026825),
 ('nonfancy', 0.7813728451728821),
 ('salsa', 0.7754536271095276),
 ('certainly', 0.7737827897071838),
 ('patio', 0.7733049988746643)]

To get one vector for each review I take the **average** of the word vectors in it. Words in the test set that are not in the vocabulary are skipped, and if a review has no known words it gets a zero vector.

In [14]:
def review_vector(tokens, model, size=200):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_vector(t, w2v_model) for t in train_tokens])
X_test_w2v = np.array([review_vector(t, w2v_model) for t in test_tokens])

print(X_train_w2v.shape, X_test_w2v.shape)

(800, 200) (200, 200)


The averaged vectors have very small values that are close to each other, so I standardized them (fit on train only). Without this the network predicted the same class for all reviews.

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_w2v = scaler.fit_transform(X_train_w2v)
X_test_w2v = scaler.transform(X_test_w2v)

In [16]:
# convert to PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print(X_train_w2v_tensor.shape, X_test_w2v_tensor.shape, y_train_tensor.shape)

torch.Size([800, 200]) torch.Size([200, 200]) torch.Size([800, 1])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [17]:
torch.manual_seed(42)

input_size = X_train_w2v_tensor.shape[1]   # 200

w2v_nn = nn.Sequential(
    nn.Linear(input_size, 128),   # hidden layer 1
    nn.ReLU(),

    nn.Linear(128, 64),           # hidden layer 2
    nn.ReLU(),

    nn.Linear(64, 32),            # hidden layer 3
    nn.ReLU(),

    nn.Linear(32, 1)              # output neuron
)

print(w2v_nn)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [18]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(w2v_nn.parameters(), lr=0.001)

With lr = 0.001 and only 15 epochs, updating the weights once per epoch (like the TF-IDF example) is only 15 updates, which is too few. So I split the training data into mini-batches of 32 reviews. It's still 15 epochs over the full data, but the weights are updated after every batch.

In [19]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train_w2v_tensor, y_train_tensor), batch_size=32, shuffle=True)

epochs = 15

for epoch in range(epochs):
    w2v_nn.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = w2v_nn(X_batch)
        loss = loss_function(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {total_loss / len(train_loader):.4f}")

Epoch 1/15 - Loss: 0.6505
Epoch 2/15 - Loss: 0.5787
Epoch 3/15 - Loss: 0.5091
Epoch 4/15 - Loss: 0.4645
Epoch 5/15 - Loss: 0.4190
Epoch 6/15 - Loss: 0.3841
Epoch 7/15 - Loss: 0.3497
Epoch 8/15 - Loss: 0.3161
Epoch 9/15 - Loss: 0.2836
Epoch 10/15 - Loss: 0.2952
Epoch 11/15 - Loss: 0.2443
Epoch 12/15 - Loss: 0.1802
Epoch 13/15 - Loss: 0.1594


Epoch 14/15 - Loss: 0.1475
Epoch 15/15 - Loss: 0.1408


In [20]:
w2v_nn.eval()

with torch.no_grad():
    logits = w2v_nn(X_test_w2v_tensor)
    probs = torch.sigmoid(logits)
    w2v_predictions = (probs >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Test Accuracy (Word2Vec): {w2v_accuracy:.3f}")

Test Accuracy (Word2Vec): 0.670


In [21]:
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"]))
print(confusion_matrix(y_test, w2v_predictions))

              precision    recall  f1-score   support

    Negative       0.62      0.80      0.70        96
    Positive       0.75      0.55      0.63       104

    accuracy                           0.67       200
   macro avg       0.69      0.68      0.67       200
weighted avg       0.69      0.67      0.67       200

[[77 19]
 [47 57]]


In [22]:
print(f"TF-IDF model accuracy:   {accuracy:.3f}")
print(f"Word2Vec model accuracy: {w2v_accuracy:.3f}")

TF-IDF model accuracy:   0.750
Word2Vec model accuracy: 0.670


### Comparison

The Word2Vec model got **67%** test accuracy, which is lower than the TF-IDF model (**75%**).

Some reasons I think this happened:
- Word2Vec was trained on only 800 short reviews, which is very little data for learning good word vectors. For example the most similar word to "good" is "prices", which doesn't really make sense.
- Averaging the word vectors loses information. A review like "not good" ends up close to "good" because the average mixes them.
- The training loss went down to 0.14 but test accuracy is only 67%, so the bigger network is overfitting the small training set.
- The model predicts negative more often (recall 0.80 for negative vs 0.55 for positive).

Using pre-trained word vectors (like Google News Word2Vec or GloVe) trained on a much bigger corpus would probably give better results.